# Demo 0 : The Databricks SQL Workspace

**Module 5A - Production SQL on Databricks (Not on the Gen AI Associate exam, but essential for production readiness)**

This notebook walks through the entire Databricks SQL experience: the SQL editor, saved queries, SQL warehouses (types, sizing, costing, optimization), alerts, query history, and system tables for auditing and cost monitoring.

Each section has **Notes** (concept + rationale) followed by a **Demo** cell with live code where applicable.

> **Why this demo exists**: The Gen AI Associate exam focuses on AI/ML, but in production you'll spend significant time in the SQL workspace. This demo ensures you're ready for real-world projects.

In [0]:
%sql
-- SETUP: Create catalog, schema, and sample data for SQL workspace demos
-- This data will be used throughout the demo to illustrate SQL editor features,
-- query history, alerts, and system table monitoring.

CREATE CATALOG IF NOT EXISTS module5a_demo0;
CREATE SCHEMA IF NOT EXISTS module5a_demo0.sql_demo;

-- Sales summary table for demos
CREATE OR REPLACE TABLE module5a_demo0.sql_demo.daily_sales (
  sale_date    DATE,
  product      STRING,
  region       STRING,
  units_sold   INT,
  revenue      DECIMAL(10,2),
  cost         DECIMAL(10,2)
);

INSERT INTO module5a_demo0.sql_demo.daily_sales VALUES
('2025-09-01', 'Pro License',     'North America', 45, 22455.00,  9000.00),
('2025-09-01', 'Enterprise',      'Europe',        12, 17988.00,  7200.00),
('2025-09-02', 'Pro License',     'Europe',        38, 18962.00,  7600.00),
('2025-09-02', 'Vector Search',   'Asia Pacific',  22,  6578.00,  2200.00),
('2025-09-03', 'Enterprise',      'North America', 18, 26982.00, 10800.00),
('2025-09-03', 'AI Gateway',      'Europe',        15,  2985.00,  1500.00),
('2025-09-04', 'Pro License',     'Asia Pacific',  30, 14970.00,  6000.00),
('2025-09-04', 'Enterprise',      'Europe',         8, 11992.00,  4800.00),
('2025-09-05', 'Pro License',     'North America', 52, 25948.00, 10400.00),
('2025-09-05', 'Vector Search',   'Europe',        18,  5382.00,  1800.00);

SELECT * FROM module5a_demo0.sql_demo.daily_sales ORDER BY sale_date;

sale_date,product,region,units_sold,revenue,cost
2025-09-01,Pro License,North America,45,22455.00,9000.00
2025-09-01,Enterprise,Europe,12,17988.00,7200.00
2025-09-02,Pro License,Europe,38,18962.00,7600.00
2025-09-02,Vector Search,Asia Pacific,22,6578.00,2200.00
2025-09-03,AI Gateway,Europe,15,2985.00,1500.00
2025-09-03,Enterprise,North America,18,26982.00,10800.00
2025-09-04,Enterprise,Europe,8,11992.00,4800.00
2025-09-04,Pro License,Asia Pacific,30,14970.00,6000.00
2025-09-05,Pro License,North America,52,25948.00,10400.00
2025-09-05,Vector Search,Europe,18,5382.00,1800.00


## The SQL Editor

### What it is
The Databricks SQL Editor is the primary interface for writing, running, and sharing SQL queries. It provides:

* **Syntax highlighting and autocomplete**: Schema-aware autocomplete for table names, column names, and SQL functions.
* **Multiple tabs**: Run multiple queries in parallel tabs, each connected to a SQL warehouse.
* **Result exploration**: View results as tables, create visualizations (bar charts, line charts, scatter plots, etc.), and download as CSV.
* **Catalog browser**: Browse Unity Catalog tables, schemas, and columns in the left sidebar.
* **Collaboration**: Share queries with teammates, comment on results, and track changes via Git folders.
* **Keyboard shortcuts**: `Cmd/Ctrl + Enter` to run, `Cmd/Ctrl + Shift + Enter` to run all.

### Key features for production
* **`_sqldf` variable**: In notebooks, SQL cell results are auto-assigned to `_sqldf` for use in subsequent Python/SQL cells.
* **Named results**: Use `%sql --name <var_name>` to name SQL results for reuse in later cells.
* **Visualizations**: Create charts directly from query results without leaving the editor.
* **Data profiles**: Generate summary statistics and visual insights for any DataFrame or table.

### The SQL sidebar
When you open the SQL tab in Databricks, you see these sections in the sidebar:

| Sidebar Item | What it does |
|---|---|
| **Editor** | Write and run SQL queries interactively |
| **Queries** | Browse and manage saved SQL queries |
| **Dashboards** | Browse and manage AI/BI dashboards |
| **Alerts** | Browse and manage SQL alerts |
| **Query History** | View execution history of all queries |
| **Warehouses** | Manage SQL warehouse compute resources |
| **Genie** | Natural language data exploration |
| **Data** | Browse Unity Catalog tables and schemas |

In [0]:
%sql --name daily_revenue_by_region
-- Demo: A typical production query you'd run in the SQL Editor
-- This query aggregates daily sales by region and would be the starting
-- point for a dashboard dataset or an alert.

SELECT
  sale_date,
  region,
  sum(units_sold)  AS total_units,
  sum(revenue)     AS total_revenue,
  sum(revenue - cost) AS gross_profit,
  round(sum(revenue - cost) / nullif(sum(revenue), 0) * 100, 1) AS margin_pct
FROM module5a_demo0.sql_demo.daily_sales
GROUP BY 1, 2
ORDER BY 1, 2;

sale_date,region,total_units,total_revenue,gross_profit,margin_pct
2025-09-01,Europe,12,17988.00,10788.00,60.0
2025-09-01,North America,45,22455.00,13455.00,59.9
2025-09-02,Asia Pacific,22,6578.00,4378.00,66.6
2025-09-02,Europe,38,18962.00,11362.00,59.9
2025-09-03,Europe,15,2985.00,1485.00,49.7
2025-09-03,North America,18,26982.00,16182.00,60.0
2025-09-04,Asia Pacific,30,14970.00,8970.00,59.9
2025-09-04,Europe,8,11992.00,7192.00,60.0
2025-09-05,Europe,18,5382.00,3582.00,66.6
2025-09-05,North America,52,25948.00,15548.00,59.9


## Saved Queries

### What they are
Saved queries are reusable SQL scripts stored in the Databricks workspace. They are version-controlled, shareable, and can be parameterized.

### Key features
* **Parameters**: Use `{{param_name}}` syntax to make queries dynamic. Users are prompted for values at run time.
* **Sharing**: Share queries with individuals or groups. Permissions are managed at the query level.
* **Scheduling**: Saved queries can be scheduled to run on a cadence (used by alerts and jobs).
* **Git integration**: Store queries in Git folders for version control and CI/CD.
* **Multiple visualizations**: One query can have multiple visualizations attached (bar, line, table, counter).
* **Lineage**: Query lineage is tracked in Unity Catalog — you can see which tables a query reads from.

### When to use saved queries vs. notebooks
| Aspect | Saved Query | Notebook |
|---|---|---|
| **Purpose** | Single SQL statement, repeated | Multi-step analysis, Python + SQL |
| **Audience** | Analysts, BI users | Data engineers, data scientists |
| **Output** | Table + visualizations | Rich text, code, charts, narrative |
| **Scheduling** | Alerts, jobs | Jobs |
| **Version control** | Git folders | Git folders |

> Saved queries are the building blocks for alerts and dashboards. Every alert is backed by a saved query.

## SQL Warehouses

### What they are
A SQL warehouse is the compute resource that executes SQL queries in Databricks SQL. It is the SQL equivalent of a cluster for notebooks, but optimized for BI and SQL workloads.

> **Databricks Free Edition note**: Since we are using Databricks Free Edition, we **cannot create new SQL warehouses**. Every free account comes with **one pre-provisioned small serverless warehouse** ("Serverless Starter Warehouse"). We will utilize that warehouse for all SQL demos in this notebook. In a production environment, you would create separate warehouses for dev/test/prod with appropriate sizing.

### Three types of SQL warehouses

| Feature | Serverless | Pro | Classic |
|---|---|---|---|
| **Photon engine** | Yes | Yes | Yes |
| **Predictive IO** | Yes | Yes | No |
| **Intelligent Workload Management** | Yes | No | No |
| **Startup time** | ~2-6 seconds | ~1-3 minutes | ~4 minutes |
| **Compute location** | Databricks account | Your AWS/Azure/GCP account | Your AWS/Azure/GCP account |
| **Use case** | BI, ETL, exploratory analysis | Custom networking, federation | Basic interactive exploration |
| **Recommended?** | Yes (default choice) | When serverless unavailable | Entry-level only |

### Why serverless is the default
* **Fast startup** (~2-6s vs ~4 min for classic): No waiting for clusters to spin up.
* **Intelligent Workload Management (IWM)**: AI-powered autoscaling that predicts query resource needs, queues intelligently, and scales rapidly.
* **Predictive IO**: Speeds up selective scans by skipping irrelevant data blocks.
* **No infrastructure to manage**: Compute lives in Databricks' account, not yours. No VPC, no instance configuration.
* **Cost-efficient at scale**: Pay only for what you use. Auto-stop shuts down idle warehouses automatically.

### Warehouse sizing
Cluster size determines compute resources per cluster. The autoscaler adds/removes clusters of that size as needed.

| Size | When to use |
|---|---|
| **X-Small** | Development, testing, small datasets |
| **Small** | Small teams, light BI workloads |
| **Medium** | Default for most BI workloads |
| **Large** | Heavy BI, large dashboards, many concurrent users |
| **X-Large** | Large datasets, complex queries, ETL |
| **2X-Large - 4X-Large** | Enterprise-scale, heavy ETL, high concurrency |

**Best practice**: Start with one larger warehouse and let serverless manage concurrency. Size down if needed. If queries spill to disk, increase cluster size. Monitor the **Peak Queued Queries** metric.

### Auto-stop and auto-scaling
* **Auto-stop**: Automatically terminates the warehouse after a configurable idle period (5 min to 10 min, or never). This is the #1 cost control for SQL warehouses.
* **Auto-scaling**: Sets min/max clusters. Serverless IWM dynamically adds clusters when queries queue and removes them when demand drops.
* **Min clusters**: Keep at 0 for cost optimization (warehouse starts on first query). Set to 1+ for latency-sensitive workloads (no cold start).
* **Max clusters**: Set high enough to handle peak concurrency. Monitor peak queued queries and adjust.

## SQL Warehouse Costing & Optimization

### Why use SQL warehouses instead of serverless notebook compute?

| Aspect | SQL Warehouse | Serverless Notebook Compute |
|---|---|---|
| **Primary use** | BI, SQL queries, dashboards, alerts | Notebooks, Python, ML, data engineering |
| **Interface** | SQL Editor, Dashboards, Genie | Notebooks, Jobs |
| **Autostop** | Yes (5 min default) | Yes (based on idle timeout) |
| **IWM** | Yes (AI-powered queue management) | No (different scheduler) |
| **Cost model** | DBU per hour, billed per second | DBU per hour, billed per second |
| **Concurrency** | Optimized for many concurrent SQL users | Optimized for interactive sessions |
| **Photon** | Yes (all types) | Yes |
| **Predictive IO** | Yes (serverless/pro only) | Yes (serverless) |

**Key takeaway**: SQL warehouses are optimized for **concurrent SQL workloads** (dashboards, alerts, many users querying simultaneously). Serverless notebook compute is optimized for **interactive Python/SQL sessions** in notebooks. For production BI, always use a SQL warehouse.

### Cost optimization strategies

1. **Auto-stop is your #1 cost control**: Set auto-stop to 5-10 min for dev/test, 15-30 min for BI workloads. A warehouse left running overnight can cost more than all your queries combined.

2. **Right-size your warehouse**: Start with Medium. If queries spill to disk, scale up. If the warehouse is always idle during business hours, scale down.

3. **Use serverless**: Serverless starts in ~5 seconds and auto-stops aggressively. Classic warehouses take ~4 min to start, encouraging users to leave them running.

4. **Min clusters = 0**: For cost optimization, keep min clusters at 0. The warehouse starts on first query. For latency-sensitive dashboards, set min to 1.

5. **Separate dev/test/prod warehouses**: Don't run a 4X-Large warehouse for development. Use X-Small for dev, Medium for test, and size up for production.

6. **Monitor with system tables**: Track DBU consumption, query latency, and warehouse events using `system.billing.usage`, `system.query.history`, and `system.compute.warehouse_events`.

### System tables for SQL monitoring

| System Table | What it tracks |
|---|---|
| `system.compute.warehouses` | Snapshots of warehouse configurations |
| `system.compute.warehouse_events` | Warehouse start, stop, scale-up, scale-down events |
| `system.query.history` | Every query executed on SQL warehouses |
| `system.billing.usage` | Billing records for all Databricks usage (DBUs) |
| `system.access.audit` | Audit log of all workspace activities |

> Every query you run on a SQL warehouse is logged in `system.query.history` with execution details, and every DBU consumed is logged in `system.billing.usage` for cost tracking.

In [0]:
%sql
-- Demo: Query SQL warehouse configurations from system tables
-- system.compute.warehouses contains snapshots of all warehouse configs
-- This shows what warehouses exist, their type, size, and auto-stop settings.

SELECT
  warehouse_id,
  warehouse_name,
  warehouse_type,
  warehouse_size,
  min_clusters,
  max_clusters,
  auto_stop_minutes,
  warehouse_channel,
  created_by
FROM system.compute.warehouses
WHERE delete_time IS NULL
ORDER BY warehouse_name
LIMIT 20;

warehouse_id,warehouse_name,warehouse_type,warehouse_size,min_clusters,max_clusters,auto_stop_minutes,warehouse_channel,created_by
cea4f31be009b0c4,Serverless Starter Warehouse,SERVERLESS,2X_SMALL,1,1,10,CURRENT,73713791015377


## Query History

### What it is
Query History is a record of every SQL query executed in your workspace. It is available both as a UI (sidebar > Query History) and as a system table (`system.query.history`).

### UI Features
* **Filter by status**: Completed, failed, running, cancelled.
* **Filter by user**: See who ran what.
* **Filter by warehouse**: Which warehouse executed the query.
* **Filter by duration**: Find slow queries.
* **Query details**: Click any query to see the full SQL text, execution plan, duration, rows returned, and warehouse used.

### Why it matters for production
* **Debug slow queries**: Find queries that take too long and optimize them.
* **Audit trail**: Every query is logged with user, timestamp, and duration — useful for compliance.
* **Cost attribution**: Join with `system.billing.usage` to estimate cost per query.
* **Identify patterns**: Find frequently run queries that could be materialized or cached.

> The `system.query.history` table includes queries from SQL warehouses, serverless compute for notebooks and jobs, and Lakeflow pipelines. It covers all workspaces in the same region.

In [0]:
%sql
-- Demo: Query your recent query history from system.query.history
-- This shows the last 10 queries you've run, with execution details.
-- Perfect for debugging slow queries or auditing who ran what.

SELECT
  statement_id AS query_id,
  statement_type,
  execution_status AS state,
  total_duration_ms / 1000.0 AS duration_seconds,
  produced_rows,
  executed_by AS user_email,
  start_time,
  substr(statement_text, 1, 80) AS query_preview
FROM system.query.history
WHERE executed_by = current_user()
ORDER BY start_time DESC
LIMIT 10;

query_id,statement_type,state,duration_seconds,produced_rows,user_email,start_time,query_preview
186addf5-5f61-44ce-bdc0-94a409976dfd,SELECT,FINISHED,0.049000,10,shriveens@platformatory.com,2026-09-25T07:45:10.817Z,"catalogs = spark.sql(""SHOW CATALOGS"").collect()"
8cb29e38-2282-445e-abc1-617f01f0a5f9,SHOW,FINISHED,0.261000,10,shriveens@platformatory.com,2026-09-25T07:45:10.520Z,"catalogs = spark.sql(""SHOW CATALOGS"").collect()"
44bcdb48-57b2-40f0-be14-ef8b89cced0e,SELECT,FINISHED,2.039000,1,shriveens@platformatory.com,2026-09-25T07:45:08.409Z,"llama4 = spark.sql("""""" SELECT ai_query( 'databricks-llama-4-maverick',"
96dd98ad-a606-4da3-afdc-81d9998d34d6,SELECT,FINISHED,1.579000,1,shriveens@platformatory.com,2026-09-25T07:45:06.501Z,"llama3 = spark.sql("""""" SELECT ai_query( 'databricks-meta-llama-3-3-70b-ins"
061f230d-9bde-4366-8eda-72a3180df843,USE,FAILED,0.179000,null,shriveens@platformatory.com,2026-09-25T07:45:05.926Z,"spark.sql(""USE CATALOG module5a_demo2"")"
8b91f8b3-3264-485a-87ca-af7360be52a2,SELECT,FINISHED,1.191000,1,shriveens@platformatory.com,2026-09-25T07:44:17.661Z,"eval_result = spark.sql(f"""""" SELECT ai_query( 'databricks-meta-llama-3-3-7"
5c72f0b6-d915-4116-9681-1b0e893fa007,SELECT,FINISHED,10.358000,3,shriveens@platformatory.com,2026-09-25T07:44:06.771Z,"result = spark.sql("""""" SELECT feedback_id, comment, module5a_demo7"
e39c72bb-67a5-4a0a-8fdd-53397f1094fa,SELECT,FINISHED,0.044000,2,shriveens@platformatory.com,2026-09-25T07:44:06.192Z,"functions = spark.sql(""SHOW USER FUNCTIONS IN ai_security"").collect()"
87278a76-2c0d-4037-88b8-0c359a61bb59,SHOW,FINISHED,0.157000,2,shriveens@platformatory.com,2026-09-25T07:44:06.001Z,"functions = spark.sql(""SHOW USER FUNCTIONS IN ai_security"").collect()"
da65d798-ca30-4b37-a2de-3887dbb1cbfb,SELECT,FINISHED,0.060000,1,shriveens@platformatory.com,2026-09-25T07:44:05.873Z,"tables = spark.sql(""SHOW TABLES IN ai_security"").collect()"


## SQL Alerts

### What they are
Databricks SQL alerts run a query on a schedule and notify you when a defined condition is met. They are the production monitoring layer for your data platform.

### How they work
1. **Write a query** that returns a single numeric value (e.g., total daily revenue).
2. **Define a condition** (e.g., revenue < 10000) and a threshold.
3. **Set a schedule** (e.g., every weekday at 9 AM).
4. **Configure notifications** (email, Slack, webhook, PagerDuty).
5. The alert runs the query on schedule, evaluates the condition, and triggers notifications.

### Alert states
| State | Meaning |
|---|---|
| **TRIGGERED** | The condition was met on the most recent run |
| **OK** | The condition was NOT met on the most recent run |
| **ERROR** | The query failed to execute |

### Common alert patterns for production

| Pattern | Example Query | Condition |
|---|---|---|
| **Business KPI monitoring** | `SELECT sum(revenue) FROM daily_sales WHERE sale_date = current_date()` | `revenue < 10000` |
| **Data quality check** | `SELECT count(*) FROM orders WHERE status IS NULL` | `NULL_count > 0` |
| **Cost spike detection** | `SELECT sum(usage_quantity) FROM system.billing.usage WHERE usage_date = current_date()` | `daily_dbus > 1000` |
| **Warehouse health** | `SELECT count(*) FROM system.query.history WHERE execution_status = 'FAILED' AND start_time > now() - INTERVAL 1 HOUR` | `failed_count > 10` |
| **Audit anomaly** | `SELECT count(*) FROM system.access.audit WHERE action_name = 'SELECT' AND event_time > now() - INTERVAL 5 MINUTE` | `select_count > 1000` |
| **Slow query detection** | `SELECT avg(total_duration_ms) FROM system.query.history WHERE start_time > now() - INTERVAL 1 HOUR` | `avg_ms > 30000` |

> Alerts can also be added as tasks in Lakeflow Jobs, enabling automated pipelines that branch based on alert results.

In [0]:
%sql
-- Demo: Alert query example - daily revenue check
-- This query would back a SQL alert that triggers if daily revenue drops below $10,000.
-- The alert would run this query on a schedule and notify the team if triggered.

SELECT
  sum(revenue) AS today_revenue
FROM module5a_demo0.sql_demo.daily_sales
WHERE sale_date = (SELECT max(sale_date) FROM module5a_demo0.sql_demo.daily_sales);

-- To create an actual alert from this query:
-- 1. Save this as a saved query
-- 2. Go to Alerts > Create Alert
-- 3. Select this query
-- 4. Set column: today_revenue, condition: <, threshold: 10000
-- 5. Set schedule: Every weekday at 9:00 AM
-- 6. Add notification destinations (email, Slack, etc.)

-- Demo: Alert query example - data quality check
-- This query checks for any sales records with NULL or negative values.
SELECT
  count(*) AS bad_records
FROM module5a_demo0.sql_demo.daily_sales
WHERE units_sold IS NULL
   OR revenue IS NULL
   OR revenue < 0
   OR cost IS NULL
   OR cost < 0;

-- Alert config: column=bad_records, condition=>, threshold=0, schedule=hourly

bad_records
0


## System Tables for Auditing and Cost Monitoring

### What system tables are
System tables are Databricks-hosted analytical store of your account's operational data. They provide historical observability across your account for usage, performance, costs, security, and compliance. They live in the `system` catalog and require admin access by default.

### Key system tables for SQL workloads

| System Table | Purpose | Key Columns |
|---|---|---|
| `system.billing.usage` | Every DBU consumed, with metadata for attribution | `usage_quantity`, `sku_name`, `usage_metadata.warehouse_id`, `custom_tags` |
| `system.billing.list_prices` | Current pricing for each SKU | `sku_name`, `pricing.default` |
| `system.query.history` | Every query executed on SQL warehouses and serverless | `statement_id`, `statement_text`, `total_duration_ms`, `executed_by` |
| `system.access.audit` | Audit log of all workspace activities | `action_name`, `event_time`, `user_identity.email`, `request_params` (map) |
| `system.compute.warehouses` | Snapshots of warehouse configurations | `warehouse_id`, `warehouse_name`, `warehouse_size`, `auto_stop_minutes` |
| `system.compute.warehouse_events` | Start/stop/scale events for warehouses | `warehouse_id`, `event_type`, `cluster_count`, `event_time` |

### Why this matters for production
* **Cost attribution**: Join `system.billing.usage` with `system.billing.list_prices` to estimate costs per warehouse, per team, per tag.
* **Audit trail**: `system.access.audit` records every SELECT, CREATE, DROP, GRANT — who did what and when. Essential for compliance (SOX, GDPR, HIPAA).
* **Query monitoring**: `system.query.history` lets you find slow queries, failed queries, and usage patterns.
* **Warehouse health**: `system.compute.warehouse_events` shows when warehouses start/stop/scale — identify wasteful patterns (e.g., warehouse running overnight).
* **Anomaly detection**: Set up alerts on system tables to catch cost spikes, unusual access patterns, or query failures.

> By default, only account admins have access to system tables. To share with non-admins, create dynamic views that filter rows by user.

In [0]:
%sql
-- Demo 1: Monitor warehouse costs by day
-- Join billing usage with list prices to estimate daily costs

SELECT
  u.usage_date,
  u.sku_name,
  round(sum(u.usage_quantity), 2) AS total_dbus,
  round(sum(u.usage_quantity * lp.pricing.default), 2) AS estimated_cost
FROM system.billing.usage AS u
LEFT JOIN system.billing.list_prices AS lp
  ON u.sku_name = lp.sku_name
  AND lp.price_end_time IS NULL
WHERE u.usage_date >= now() - INTERVAL 7 DAY
  AND u.usage_metadata.warehouse_id IS NOT NULL
GROUP BY 1, 2
ORDER BY 1 DESC, 2;

usage_date,sku_name,total_dbus,estimated_cost


In [0]:
%sql
-- Demo 2: Audit log - recent SQL queries by current user
-- system.access.audit records every action in the workspace
-- This shows recent SELECT and CREATE actions for audit purposes

SELECT
  event_time,
  action_name,
  user_identity.email AS user,
  request_params['full_name'] AS target_object,
  source_ip_address
FROM system.access.audit
WHERE user_identity.email = current_user()
  AND action_name IN ('SELECT', 'CREATE', 'DROP', 'GRANT')
  AND event_time >= now() - INTERVAL 1 DAY
ORDER BY event_time DESC
LIMIT 20;

event_time,action_name,user,target_object,source_ip_address


In [0]:
%sql
-- Demo 3: Warehouse events - when did warehouses start/stop/scale?
-- This helps identify wasteful patterns (e.g., warehouse running overnight)

SELECT
  event_time,
  warehouse_id,
  event_type,
  cluster_count
FROM system.compute.warehouse_events
WHERE event_time >= now() - INTERVAL 7 DAY
ORDER BY event_time DESC
LIMIT 20;

event_time,warehouse_id,event_type,cluster_count


## Dashboards and Genie (Brief)

> **Full coverage**: See [Demo 1 : Dataset-to-Genie Walkthrough](#notebook-1450212875215944) for hands-on dashboards and Genie.

### Dashboards (overview)
AI/BI Dashboards are visual, interactive reports built on UC-governed data. Key building blocks: datasets (SQL queries), canvas, visualizations, filters, and pages. Build one well-designed dataset per page, then create multiple visualizations from it.

### Genie Agent (overview)
Genie is a natural-language interface to governed data tables. Users ask questions in English, and Genie generates and runs SQL automatically. Best for ad-hoc, exploratory questions that dashboards don't answer. Add Genie Space instructions to define key metrics, table descriptions, and example questions.

### When to use what
* **Dashboard**: Known, repeated questions, many users, visual format needed.
* **Genie**: Ad-hoc, exploratory questions, individual users, flexibility matters.
* **Saved Query**: Single SQL statement you run repeatedly, no visualization needed.
* **Alert**: Monitor a metric on a schedule and get notified when it crosses a threshold.

## Learning Conclusion

### What we covered

| Topic | What was covered | Key Takeaway |
|---|---|---|
| SQL Editor | Writing queries, autocomplete, visualizations, `_sqldf` | The primary interface for SQL on Databricks |
| Saved Queries | Reusable SQL with parameters, sharing, scheduling | Building blocks for alerts and dashboards |
| SQL Warehouses | Serverless vs Pro vs Classic; sizing; auto-stop; IWM | Serverless is the default: fast start, AI-managed scaling, cost-efficient |
| Costing & Optimization | Auto-stop, right-sizing, min clusters, system tables | Auto-stop is #1 cost control; monitor with system tables |
| Query History | UI + `system.query.history` for every executed query | Audit trail, debug slow queries, cost attribution |
| Alerts | Scheduled query + condition + notification | Monitor KPIs, data quality, cost spikes, warehouse health |
| System Tables | Billing, audit, query history, warehouse events | Full observability: who, what, when, how much |
| Dashboards & Genie | Brief overview (see Demo 1) | Dashboards for known questions; Genie for ad-hoc |

### Key principles for production SQL
* **Always use SQL warehouses for BI workloads** (not notebook compute). They are optimized for concurrent SQL queries.
* **Serverless + auto-stop = cost efficiency**. Serverless starts in ~5s, auto-stop kills idle warehouses in minutes.
* **Everything is logged**: Every query, every DBU, every action — all in system tables. Use them for monitoring, auditing, and cost optimization.
* **Alerts are your monitoring layer**: Set up alerts on system tables for cost spikes, query failures, and data quality.
* **UC governance applies everywhere**: Tables, queries, dashboards, and Genie all respect Unity Catalog permissions.

In [0]:
%sql
-- CLEANUP: Decommission everything created in this demo
DROP TABLE IF EXISTS module5a_demo0.sql_demo.daily_sales;
DROP SCHEMA IF EXISTS module5a_demo0.sql_demo CASCADE;
DROP CATALOG IF EXISTS module5a_demo0 CASCADE;

SELECT 'Cleanup complete!' AS status;